<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/prompt_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

In [ ]:
!pip install dspy

In [ ]:
import pandas as pd
import dspy

In [ ]:
training_df = pd.read_excel('Dungeons-and-Dragons-Turn-Classification/pipeline_testing_set.xlsx')

In [ ]:
training_df

In [ ]:
training_df.drop(columns=['Index'], inplace=True)

In [ ]:
training_df.drop(columns=['Unnamed: 4'], inplace=True)

In [ ]:
training_df

In [ ]:
trainset = []

for context, current_turn, category in training_df.values:
    examples = dspy.Example(context=context, question=current_turn, response=category).with_inputs("context", "question")
    trainset.append(examples)

In [ ]:
trainset[:5]

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434', api_key='', max_tokens=2048)
dspy.configure(lm=lm)

In [ ]:
from typing import Literal


class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['Gameplay Mechanic',
                      'Out-Of-Game Conversation',
                      'Worldbuilding',
                      'Strategising',
                      ] = dspy.OutputField()

class PlayerInstruction(dspy.Signature):
  category: Literal['Gameplay Mechanic',
                    'Out-Of-Game Conversation',
                    'Worldbuilding',
                    'Strategising',
                    ] = dspy.InputField()
  player_instruction:str = dspy.OutputField(desc="instruction on how to behave within a D&D game.")

In [ ]:
class ClassifyTurns(dspy.Module):
  def __init__(self):
    self.classifier = dspy.ChainOfThought(TurnClassifier, caching=False)

  def forward(self, context, question, **kwargs):
    prediction = self.classifier(context=context, question=question)
    return prediction


In [ ]:
classify = ClassifyTurns()
classify.load("optimized_classifier.json")

In [ ]:
class PromptGenerator(dspy.Module):
  def __init__(self):
    self.classifier = classify
    self.generator = dspy.ChainOfThought(PlayerInstruction, caching=False)

  def forward(self, context, question, **kwargs):
    pred_category = classify(context=context, question=question).response
    prompt = self.generator(category=pred_category)
    return prompt, pred_category

prompt_generator = PromptGenerator()


In [ ]:
prompt_generator(trainset[1]['context'], trainset[1]['question'])

In [ ]:
generated_prompt = []

for i in range(len(trainset[0:20])):
    generated_prompt.append(prompt_generator(trainset[i]['context'], trainset[i]['question']))

In [ ]:
generated_prompt

In [ ]:
generated_prompt[1][0]['player_instruction']

In [ ]:
category = []
player_instruction = []

for i in range(len(generated_prompt)):
  category.append(generated_prompt[i][1])
  player_instruction.append(generated_prompt[i][0]['player_instruction'])


In [ ]:
category

In [ ]:
player_instruction

In [ ]:
instruction_library = list(zip(category, player_instruction))

In [ ]:
instruction_library

In [ ]:
instruction_df = pd.DataFrame(instruction_library)

In [ ]:
instruction_df.drop_duplicates(subset=[0], inplace=False)

In [ ]:
filename = 'instructions.xlsx'
instruction_df.to_excel(filename)